# Lesson 7: Testing AI Systems

## WHY
Your moderation pipeline from Lesson 6 works, but how do you know it still works after a refactor?
How do you verify that a "phishing link" always triggers a ban, not just a warning?
And how do you test without spending money on API calls?

Testing AI systems is different from testing normal code:
- **LLM outputs are non-deterministic** — the same input can produce different text each time
- **API calls are expensive** — running 50 test cases at $0.002 each costs $0.10 per `pytest` run, and it adds up
- **API calls are slow** — each test takes 1-3 seconds, making your test suite painfully sluggish
- **API calls can fail** — rate limits, outages, and network errors make tests flaky

The solution: **fake models**. LangChain provides `FakeListChatModel` — a chat model that returns
pre-scripted responses instead of calling an API. You control exactly what the "LLM" says,
so your tests are fast, free, deterministic, and offline.

**By the end of this notebook you will:**
1. Use `FakeListChatModel` to simulate LLM responses without API calls
2. Test structured output with fake models
3. Test agents created with `create_agent` using fake models
4. Write async tests with `pytest-asyncio`
5. Mock Discord objects for handler testing
6. Understand the testing pyramid for AI applications

## Setup

In [ ]:
# No API key needed for this lesson! That's the whole point.
# FakeListChatModel doesn't call any external API.
print("No API key required — all tests use fake models")

## WHAT — `FakeListChatModel`

`FakeListChatModel` is a LangChain chat model that cycles through a list of pre-defined responses.
It has the same interface as `ChatOpenAI` — you can call `.invoke()`, `.ainvoke()`, `.bind_tools()`,
and `.with_structured_output()` on it.

```python
from langchain_core.language_models import FakeListChatModel

fake_llm = FakeListChatModel(responses=["First response", "Second response"])
fake_llm.invoke("anything")  # → AIMessage(content="First response")
fake_llm.invoke("anything")  # → AIMessage(content="Second response")
fake_llm.invoke("anything")  # → AIMessage(content="First response")  # cycles
```

Key properties:
- **Free** — no API calls, no tokens, no cost
- **Fast** — returns instantly, no network latency
- **Deterministic** — same responses every time
- **Offline** — works without internet

📖 [FakeListChatModel reference](https://python.langchain.com/docs/how_to/test_with_fake_models/)

In [ ]:
from langchain_core.language_models import FakeListChatModel
from langchain_core.messages import HumanMessage

# Create a fake model with pre-scripted responses
fake_llm = FakeListChatModel(responses=[
    "This message looks fine to me.",
    "This is definitely spam — take action.",
])

# Call it — returns responses in order, then cycles
r1 = fake_llm.invoke([HumanMessage(content="Check this message")])
print(f"Call 1: {r1.content}")

r2 = fake_llm.invoke([HumanMessage(content="Check another message")])
print(f"Call 2: {r2.content}")

r3 = fake_llm.invoke([HumanMessage(content="And another")])
print(f"Call 3: {r3.content}  ← cycles back to first")

In [ ]:
# ainvoke works too — important since our moderation pipeline is async
r = await fake_llm.ainvoke([HumanMessage(content="async test")])
print(f"Async call: {r.content}")

## HOW — Testing Structured Output with Fake Models

`.with_structured_output()` works with `FakeListChatModel`, but you need to provide responses
that are valid JSON matching your Pydantic schema. The fake model returns the string, and
the structured output wrapper parses it into your Pydantic model.

In [ ]:
import json
from typing import Literal
from pydantic import BaseModel, Field


class ModerationVerdict(BaseModel):
    """A structured moderation decision."""

    action: Literal["allow", "warn", "mute", "kick", "ban"] = Field(
        description="The moderation action to take"
    )
    reasoning: str = Field(
        description="Step-by-step explanation"
    )
    confidence: float = Field(
        description="Confidence from 0.0 to 1.0"
    )


# Prepare a JSON response that matches the schema
fake_verdict = json.dumps({
    "action": "ban",
    "reasoning": "Phishing link detected — severe violation.",
    "confidence": 0.95,
})

print(f"Fake response JSON: {fake_verdict}")

In [ ]:
# Wrap the fake model with structured output
fake_structured = FakeListChatModel(
    responses=[fake_verdict]
).with_structured_output(ModerationVerdict)

# Invoke — returns a ModerationVerdict, not an AIMessage
verdict = fake_structured.invoke([HumanMessage(content="test")])

print(f"Type: {type(verdict)}")
print(f"Action: {verdict.action}")
print(f"Confidence: {verdict.confidence}")
print(f"Reasoning: {verdict.reasoning}")

# This is what you'd assert in a test:
assert verdict.action == "ban"
assert verdict.confidence == 0.95
print("\n✓ Assertions passed!")

### Testing the Triage Model

The triage model from Lesson 6 uses `.with_structured_output(QuickTriageVerdict)`.
Let's test it with different fake responses.

In [ ]:
class QuickTriageVerdict(BaseModel):
    """Lightweight triage result."""

    flagged: bool = Field(description="True if needs review")
    reason: str = Field(description="Why flagged or passed")


# Test case: message should pass triage
safe_response = json.dumps({"flagged": False, "reason": "Normal question"})
fake_triage_safe = FakeListChatModel(
    responses=[safe_response]
).with_structured_output(QuickTriageVerdict)

triage = fake_triage_safe.invoke([HumanMessage(content="test")])
assert triage.flagged is False
print(f"Safe triage: flagged={triage.flagged}, reason={triage.reason}")

# Test case: message should be flagged
flagged_response = json.dumps({"flagged": True, "reason": "Contains phishing link"})
fake_triage_flag = FakeListChatModel(
    responses=[flagged_response]
).with_structured_output(QuickTriageVerdict)

triage = fake_triage_flag.invoke([HumanMessage(content="test")])
assert triage.flagged is True
print(f"Flagged triage: flagged={triage.flagged}, reason={triage.reason}")

print("\n✓ Both triage test cases passed!")

## HOW — Testing Agents with `create_agent`

`create_agent` accepts **either** a model string (`"openai:gpt-4o-mini"`) **or** a model instance.
Model strings won't work with `FakeListChatModel` — you need to pass the fake model instance directly.

```python
# Production code — model string
agent = create_agent("openai:gpt-4o-mini", tools=my_tools)

# Test code — fake model instance
fake = FakeListChatModel(responses=[...])
agent = create_agent(fake, tools=my_tools)
```

This is a key design benefit of `create_agent` — swapping models is trivial.

**Important:** When testing agents with tools, the fake model needs to return responses that
simulate the agent's ReAct loop:
1. First response: a tool call (as JSON/AIMessage with tool_calls)
2. Subsequent responses: more tool calls or a final text answer

In [ ]:
from langchain.agents import create_agent
from langchain_core.tools import tool


# Simple tools for testing
@tool
def get_server_rules(category: str = "all") -> str:
    """Look up server rules."""
    return "No spam, no phishing, be respectful."


@tool
def get_user_warn_history(user_id: str) -> str:
    """Look up user's warning history."""
    return f"User {user_id}: 2 previous warnings"


test_tools = [get_server_rules, get_user_warn_history]
print(f"Test tools: {[t.name for t in test_tools]}")

### Simulating Tool Calls with FakeListChatModel

The tricky part of testing agents is simulating the tool-call protocol.
The fake model needs to return messages that look like the LLM requested tool calls.

LangChain's `FakeListChatModel` cycles through string responses. For agents, the simplest approach is
to test with a final text response (no tool calls) and verify the agent returns cleanly.

For more advanced testing, you can use `GenericFakeChatModel` which supports custom message sequences
including tool calls — but that's beyond what we need here.

In [ ]:
# Simple agent test: fake model returns a direct answer (no tool calls)
# This tests that the agent handles the "model decides not to use tools" path
fake_agent_llm = FakeListChatModel(
    responses=["The message is fine. No rules violated."]
)

test_agent = create_agent(
    fake_agent_llm,  # pass instance, not string
    tools=test_tools,
    system_prompt="You are a moderation bot.",
)

result = await test_agent.ainvoke({
    "messages": [HumanMessage(content="User said: 'Hello everyone'")]
})

final_message = result["messages"][-1].content
print(f"Agent response: {final_message}")
assert "fine" in final_message.lower() or "no rules" in final_message.lower()
print("✓ Agent test passed!")

## HOW — Testing with `response_format=`

When the agent uses `response_format=`, the final LLM call produces structured output.
With a fake model, you provide the JSON string that represents your structured response.

The agent makes multiple LLM calls during its ReAct loop, so you need to provide a sequence:
responses for the reasoning steps, then the final structured JSON.

In [ ]:
# For response_format agents, the last response needs to be valid JSON
# matching the schema. The agent uses it for the structured_response.
structured_fake_responses = [
    # Response 1: Agent's initial reasoning (no tool calls → goes to END)
    "This message contains a phishing link. I should ban this user.",
    # Response 2: The structured output formatting call
    json.dumps({
        "action": "ban",
        "reasoning": "Phishing link detected — severe violation regardless of history.",
        "confidence": 0.95,
    }),
]

fake_structured_llm = FakeListChatModel(responses=structured_fake_responses)

structured_agent = create_agent(
    fake_structured_llm,
    tools=test_tools,
    system_prompt="You are a moderation bot.",
    response_format=ModerationVerdict,
)

result = await structured_agent.ainvoke({
    "messages": [HumanMessage(content="User posted a scam link")]
})

verdict = result["structured_response"]
print(f"Action: {verdict.action}")
print(f"Confidence: {verdict.confidence}")
assert verdict.action == "ban"
assert verdict.confidence == 0.95
print("\n✓ Structured agent test passed!")

## WHAT — The Testing Pyramid for AI Applications

AI applications have a unique testing pyramid:

```
         /\          Eval suite (real model, real prompts)
        /  \         Slow, expensive — run weekly or on major changes
       /    \
      /------\       Integration tests (fake model, full pipeline)
     / Unit   \      Fast — run on every commit
    /  Tests    \
   /____________\    Unit tests (individual functions, no LLM)
                     Instant — run constantly
```

| Layer | What you test | Model | Speed | Cost |
|-------|--------------|-------|-------|------|
| **Unit** | `handle_verdict()`, `should_moderate()`, Pydantic models | None | Instant | Free |
| **Integration** | Full pipeline: triage → agent → action | `FakeListChatModel` | Fast | Free |
| **Eval** | Prompt quality, edge cases, regression | Real `ChatOpenAI` | Slow | $$$ |

This lesson covers the first two layers. Evaluations (the top layer) are covered in Lesson 15 (LangSmith).

## HOW — Unit Tests (No LLM Required)

These test pure Python logic — no LLM, no fake model, just functions.

In [ ]:
from pydantic import ValidationError


# ── Test 1: ModerationVerdict validates correctly ──────────────

def test_verdict_valid():
    v = ModerationVerdict(action="ban", reasoning="test", confidence=0.9)
    assert v.action == "ban"
    assert v.confidence == 0.9


def test_verdict_rejects_invalid_action():
    try:
        ModerationVerdict(action="delete", reasoning="test", confidence=0.5)
        assert False, "Should have raised ValidationError"
    except ValidationError:
        pass  # expected


test_verdict_valid()
test_verdict_rejects_invalid_action()
print("✓ Pydantic model tests passed!")

In [ ]:
# ── Test 2: handle_verdict branching (from Lesson 5) ──────────

def handle_verdict(verdict: ModerationVerdict) -> str:
    """Take action based on a verdict."""
    match verdict.action:
        case "allow":
            return "allowed"
        case "warn":
            return f"warned: {verdict.reasoning}"
        case "mute":
            return f"muted: {verdict.reasoning}"
        case "kick":
            return f"kicked: {verdict.reasoning}"
        case "ban":
            return f"banned: {verdict.reasoning}"


def test_handle_allow():
    v = ModerationVerdict(action="allow", reasoning="Clean message", confidence=0.99)
    assert handle_verdict(v) == "allowed"


def test_handle_ban():
    v = ModerationVerdict(action="ban", reasoning="Phishing", confidence=0.95)
    result = handle_verdict(v)
    assert result.startswith("banned:")
    assert "Phishing" in result


test_handle_allow()
test_handle_ban()
print("✓ handle_verdict tests passed!")

In [ ]:
# ── Test 3: should_moderate pre-filter ─────────────────────────

MIN_MESSAGE_LENGTH = 5


async def should_moderate(user_id: str, message_content: str, bot_user_id: str) -> bool:
    """Quick checks before spending tokens."""
    if user_id == bot_user_id:
        return False
    if len(message_content.strip()) < MIN_MESSAGE_LENGTH:
        return False
    if not message_content.strip():
        return False
    return True


# Async test — works in notebook cells
assert await should_moderate("user_1", "hi", "bot") is False        # too short
assert await should_moderate("bot", "spam spam spam", "bot") is False  # bot's own
assert await should_moderate("user_1", "", "bot") is False           # empty
assert await should_moderate("user_1", "Check this link!", "bot") is True  # valid
print("✓ should_moderate tests passed!")

## HOW — Integration Tests (Fake Model, Full Pipeline)

Integration tests verify the entire moderation pipeline works end-to-end,
but with fake models instead of real API calls.

In [ ]:
import logging
import time
from langchain_core.messages import HumanMessage, SystemMessage

logger = logging.getLogger("moderation")

TRIAGE_SYSTEM_PROMPT = (
    "You are a fast message triage system.\n"
    "Quickly decide if a message MIGHT violate Discord rules."
)


async def run_moderation_agent(agent, user_id, message_content):
    """Run the full moderation agent (from Lesson 6)."""
    start = time.monotonic()
    try:
        result = await agent.ainvoke(
            {"messages": [HumanMessage(content=(
                f"User {user_id} posted: '{message_content}'\n"
                "Evaluate and take action if needed."
            ))]},
            config={"recursion_limit": 15},
        )
        verdict = result["structured_response"]
        logger.info("Verdict for %s: %s", user_id, verdict.action)
        return verdict
    except Exception:
        logger.error("Agent failed for %s", user_id, exc_info=True)
        return None


async def moderate_message(triage_model, agent, user_id, message_content):
    """Two-stage moderation (from Lesson 6)."""
    try:
        triage = await triage_model.ainvoke([
            SystemMessage(content=TRIAGE_SYSTEM_PROMPT),
            HumanMessage(content=f"User {user_id} posted: '{message_content}'"),
        ])
        logger.info("Triage for %s: flagged=%s", user_id, triage.flagged)
    except Exception:
        logger.error("Triage failed", exc_info=True)
        return None

    if not triage.flagged:
        return None

    return await run_moderation_agent(agent, user_id, message_content)


print("Pipeline functions defined")

In [ ]:
# ── Integration Test: Message passes triage (not flagged) ─────

async def test_safe_message_passes_triage():
    """Safe messages should pass triage and return None (no agent cost)."""
    fake_triage = FakeListChatModel(
        responses=[json.dumps({"flagged": False, "reason": "Normal question"})]
    ).with_structured_output(QuickTriageVerdict)

    # Agent should never be called, but we create one anyway
    fake_agent = create_agent(
        FakeListChatModel(responses=["Should not reach here"]),
        tools=test_tools,
        system_prompt="test",
    )

    result = await moderate_message(
        fake_triage, fake_agent,
        user_id="user_789",
        message_content="Can someone help with Python?",
    )

    assert result is None, "Safe message should return None"
    print("✓ Safe message correctly passed triage")


await test_safe_message_passes_triage()

In [ ]:
# ── Integration Test: Flagged message triggers agent ──────────

async def test_flagged_message_gets_verdict():
    """Flagged messages should go through the full agent and return a verdict."""
    fake_triage = FakeListChatModel(
        responses=[json.dumps({"flagged": True, "reason": "Possible spam"})]
    ).with_structured_output(QuickTriageVerdict)

    fake_agent_model = FakeListChatModel(responses=[
        "This is a phishing link. Ban the user.",
        json.dumps({
            "action": "ban",
            "reasoning": "Phishing link — severe violation.",
            "confidence": 0.95,
        }),
    ])

    fake_agent = create_agent(
        fake_agent_model,
        tools=test_tools,
        system_prompt="Moderate messages.",
        response_format=ModerationVerdict,
    )

    result = await moderate_message(
        fake_triage, fake_agent,
        user_id="user_123",
        message_content="FREE NITRO! Click: scam.com",
    )

    assert result is not None, "Flagged message should return a verdict"
    assert result.action == "ban"
    assert result.confidence == 0.95
    print(f"✓ Flagged message got verdict: {result.action} (confidence: {result.confidence})")


await test_flagged_message_gets_verdict()

In [ ]:
# ── Integration Test: Agent failure returns None ───────────────

async def test_agent_failure_returns_none():
    """If the agent crashes, moderate_message should return None (fail-open)."""
    fake_triage = FakeListChatModel(
        responses=[json.dumps({"flagged": True, "reason": "Suspicious"})]
    ).with_structured_output(QuickTriageVerdict)

    # Agent model returns invalid JSON for structured output → parse error
    fake_agent_model = FakeListChatModel(responses=[
        "Thinking about this...",
        "not valid json at all {{{{",  # will fail structured output parsing
    ])

    fake_agent = create_agent(
        fake_agent_model,
        tools=test_tools,
        system_prompt="test",
        response_format=ModerationVerdict,
    )

    result = await moderate_message(
        fake_triage, fake_agent,
        user_id="user_456",
        message_content="Something suspicious",
    )

    assert result is None, "Agent failure should return None"
    print("✓ Agent failure correctly returned None (fail-open)")


await test_agent_failure_returns_none()

## HOW — Writing `pytest` Tests

The tests above run in notebook cells. For the real project, you'd put them in `tests/` and run with `uv run pytest`.

Since our moderation pipeline is async, you need `pytest-asyncio`. It lets you write `async def test_...()` functions.

```bash
uv add --dev pytest-asyncio
```

Then annotate async tests with `@pytest.mark.asyncio`:

```python
import pytest

@pytest.mark.asyncio
async def test_safe_message_passes_triage():
    # ... same test as above ...
    assert result is None
```

In [ ]:
# Here's what a complete test file would look like:
# Save this as tests/test_moderation.py

test_file_content = '''
"""Tests for the moderation pipeline."""

import json

import pytest
from langchain.agents import create_agent
from langchain_core.language_models import FakeListChatModel
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from pydantic import BaseModel, Field, ValidationError
from typing import Literal


# ── Models under test ─────────────────────────────────────────

class QuickTriageVerdict(BaseModel):
    flagged: bool = Field(description="True if needs review")
    reason: str = Field(description="Why flagged or passed")


class ModerationVerdict(BaseModel):
    action: Literal["allow", "warn", "mute", "kick", "ban"] = Field(
        description="The moderation action"
    )
    reasoning: str = Field(description="Explanation")
    confidence: float = Field(description="Confidence 0-1")


# ── Fixtures ──────────────────────────────────────────────────

@tool
def get_server_rules(category: str = "all") -> str:
    """Look up server rules."""
    return "No spam, no phishing."


@pytest.fixture
def test_tools():
    return [get_server_rules]


# ── Unit Tests ────────────────────────────────────────────────

def test_verdict_valid():
    v = ModerationVerdict(action="ban", reasoning="test", confidence=0.9)
    assert v.action == "ban"


def test_verdict_rejects_invalid_action():
    with pytest.raises(ValidationError):
        ModerationVerdict(action="delete", reasoning="test", confidence=0.5)


# ── Async Integration Tests ──────────────────────────────────

@pytest.mark.asyncio
async def test_triage_passes_safe_message():
    fake_triage = FakeListChatModel(
        responses=[json.dumps({"flagged": False, "reason": "Normal"})]
    ).with_structured_output(QuickTriageVerdict)

    result = await fake_triage.ainvoke(
        [HumanMessage(content="Can someone help with Python?")]
    )
    assert result.flagged is False


@pytest.mark.asyncio
async def test_structured_agent_returns_verdict(test_tools):
    fake_model = FakeListChatModel(responses=[
        "Looks like spam.",
        json.dumps({
            "action": "ban",
            "reasoning": "Phishing detected.",
            "confidence": 0.95,
        }),
    ])

    agent = create_agent(
        fake_model,
        tools=test_tools,
        system_prompt="Moderate messages.",
        response_format=ModerationVerdict,
    )

    result = await agent.ainvoke({
        "messages": [HumanMessage(content="SCAM LINK")]
    })

    verdict = result["structured_response"]
    assert verdict.action == "ban"
    assert verdict.confidence == 0.95
'''

print(test_file_content)

### Running the Tests

```bash
# Install pytest-asyncio
uv add --dev pytest-asyncio

# Run all tests
uv run pytest

# Run with verbose output
uv run pytest -v

# Run only moderation tests
uv run pytest tests/test_moderation.py -v
```

Add `asyncio_mode = "auto"` to `pyproject.toml` so you don't need `@pytest.mark.asyncio` on every test:

```toml
[tool.pytest.ini_options]
asyncio_mode = "auto"
```

## HOW — Mocking Discord Objects

For tests that interact with Discord (like `on_message()`), you need to mock Discord objects.
Python's `unittest.mock` provides everything you need.

The goal: create fake `discord.Message` and `discord.Member` objects that behave enough
like the real thing for your handler to work.

In [ ]:
from unittest.mock import AsyncMock, MagicMock


def make_fake_message(
    content: str,
    author_id: int = 12345,
    author_name: str = "TestUser",
    bot: bool = False,
) -> MagicMock:
    """Create a fake discord.Message for testing.

    Parameters
    ----------
    content : str
        The message text.
    author_id : int
        The fake author's user ID.
    author_name : str
        The fake author's display name.
    bot : bool
        Whether the author is a bot.

    Returns
    -------
    MagicMock
        A mock discord.Message with the essential attributes.
    """
    message = MagicMock()
    message.content = content
    message.author.id = author_id
    message.author.display_name = author_name
    message.author.bot = bot
    message.channel.name = "general"
    message.guild.name = "Test Server"

    # Mock async methods that the handler might call
    message.author.timeout_for = AsyncMock()
    message.author.ban = AsyncMock()
    message.delete = AsyncMock()
    message.channel.send = AsyncMock()

    return message


# Test the mock
msg = make_fake_message("Hello everyone", author_id=42, author_name="Alice")
print(f"Content: {msg.content}")
print(f"Author: {msg.author.display_name} (ID: {msg.author.id})")
print(f"Channel: {msg.channel.name}")
print(f"Is bot: {msg.author.bot}")

In [ ]:
# You can verify that async Discord methods were called
await msg.author.ban(reason="Testing")
msg.author.ban.assert_called_once_with(reason="Testing")
print("✓ Ban mock was called correctly")

await msg.channel.send("You've been warned.")
msg.channel.send.assert_called_once_with("You've been warned.")
print("✓ Channel send mock was called correctly")

### Using Mocks in Handler Tests

Here's how you'd test the complete `on_message` flow with mocked Discord objects.

In [ ]:
async def test_on_message_bans_phishing():
    """Phishing messages should result in a ban."""
    # Set up fakes
    fake_triage = FakeListChatModel(
        responses=[json.dumps({"flagged": True, "reason": "Phishing link"})]
    ).with_structured_output(QuickTriageVerdict)

    fake_agent_model = FakeListChatModel(responses=[
        "Phishing detected.",
        json.dumps({
            "action": "ban",
            "reasoning": "Phishing link — severe violation.",
            "confidence": 0.98,
        }),
    ])
    fake_agent = create_agent(
        fake_agent_model,
        tools=test_tools,
        system_prompt="Moderate.",
        response_format=ModerationVerdict,
    )

    # Run the pipeline
    verdict = await moderate_message(
        fake_triage, fake_agent,
        user_id="user_123",
        message_content="FREE NITRO! Click: scam.com",
    )

    # Verify verdict
    assert verdict is not None
    assert verdict.action == "ban"

    # Simulate what the bot would do with this verdict
    msg = make_fake_message("FREE NITRO!", author_id=123)
    if verdict.action == "ban":
        await msg.author.ban(reason=verdict.reasoning)

    # Verify the Discord API was "called"
    msg.author.ban.assert_called_once()
    print(f"✓ Phishing message resulted in ban (confidence: {verdict.confidence})")


await test_on_message_bans_phishing()

## Deep Dive — What to Test and What Not To

### Test with fake models:
- Pipeline wiring (does triage → agent → action work?)
- Error handling (does the pipeline survive failures?)
- Business logic branching (ban for phishing, warn for first offense, etc.)
- Structured output parsing (does valid JSON become the right Pydantic model?)

### Don't test with fake models:
- **Prompt quality** — fake models ignore your prompt entirely. Use real models + LangSmith evals (Lesson 15)
- **Tool selection** — whether the LLM picks the right tool. That's prompt engineering, not code logic
- **LLM behavior** — whether gpt-4o-mini is "better" than gpt-4o at moderation. That's evaluation, not testing

The rule of thumb: **test your code, evaluate your prompts**.

## Summary

| Concept | Key takeaway |
|---------|-------------|
| `FakeListChatModel` | Returns pre-scripted responses — free, fast, deterministic, offline |
| Structured output testing | Provide valid JSON strings as fake responses; `.with_structured_output()` parses them |
| Agent testing | Pass `FakeListChatModel` *instance* to `create_agent` (not a model string) |
| `response_format=` testing | Provide multiple responses: reasoning text + JSON for the structured output |
| `pytest-asyncio` | `@pytest.mark.asyncio` + `async def test_...()` for testing async code |
| `asyncio_mode = "auto"` | Add to `pyproject.toml` to skip the marker on every test |
| `MagicMock` / `AsyncMock` | Fake Discord objects with verifiable method calls |
| Testing pyramid | Unit (no LLM) → Integration (fake LLM) → Eval (real LLM) |
| What to test | Pipeline wiring, error handling, business logic, structured parsing |
| What NOT to test | Prompt quality, tool selection, LLM behavior — those are evals, not tests |

**Next up → Module 5, Lesson 8:** LangGraph fundamentals — build the moderation workflow as a manual `StateGraph` to understand what `create_agent` does under the hood.